# Uni-Mol Fine-Tuning for Binding Affinity Prediction

This notebook fine-tunes **Uni-Mol v2** on LP-PDBBind for protein-ligand
binding affinity prediction (pKa regression), then compares against our
custom EGNN predictor.

**Requirements:** Colab Pro with A100 GPU runtime.

## 1. Install Dependencies

In [ ]:
!pip install -q unimol_tools rdkit-pypi pandas scikit-learn matplotlib

## 2. Prepare LP-PDBBind Data

We load the LP-PDBBind refined set and convert it to the CSV format
Uni-Mol expects: `SMILES, TARGET` columns.

In [ ]:
import pandas as pd
import os

# Download LP-PDBBind index (SMILES + pKa labels)
# If you have the full dataset locally, upload the index CSV.
# Otherwise, we use the PDBBind v2020 refined set index.
INDEX_URL = (
    "https://raw.githubusercontent.com/deepmodeling/Uni-Mol/"
    "main/unimol/example_data/molecule/train.csv"
)

# For the actual project, replace with your LP-PDBBind data:
# df = pd.read_csv('path/to/lp_pdbbind_index.csv')
# df = df[['SMILES', 'pKa']].rename(columns={'pKa': 'TARGET'})

# Demo: generate a synthetic dataset from known drugs + random pKa
# Replace this block with real LP-PDBBind data for production training
import numpy as np
from rdkit import Chem

np.random.seed(42)

KNOWN_DRUGS = [
    'CC(=O)OC1=CC=CC=C1C(=O)O',         # aspirin
    'CN1C=NC2=C1C(=O)N(C(=O)N2C)C',     # caffeine
    'CC(C)Cc1ccc(cc1)[C@@H](C)C(=O)O',  # ibuprofen
    'CC(C)CC1=CC=C(C=C1)C(C)C(=O)O',    # ibuprofen isomer
    'C1=CC=C(C=C1)O',                    # phenol
    'CCO',                                # ethanol
    'c1ccc2c(c1)cc1ccc3cccc4ccc2c1c34',  # pyrene
    'CC(=O)NC1=CC=C(C=C1)O',             # acetaminophen
    'OC(=O)C1=CC=CC=C1O',               # salicylic acid
    'CC12CCC3C(C1CCC2O)CCC4=CC(=O)CCC34C', # testosterone
]

# Generate more SMILES by enumerating from a small fragment library
smiles_list = KNOWN_DRUGS * 50  # repeat for demo
pka_values = np.random.normal(6.5, 1.5, len(smiles_list)).clip(2, 12)

df = pd.DataFrame({'SMILES': smiles_list, 'TARGET': pka_values})
df = df.drop_duplicates(subset='SMILES').reset_index(drop=True)

print(f'Dataset: {len(df)} molecules')
print(f'pKa range: {df.TARGET.min():.1f} - {df.TARGET.max():.1f}')
df.head()

In [ ]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=42,
)

os.makedirs('data', exist_ok=True)
train_df.to_csv('data/unimol_train.csv', index=False)
test_df.to_csv('data/unimol_test.csv', index=False)

print(f'Train: {len(train_df)}, Test: {len(test_df)}')

## 3. Fine-Tune Uni-Mol

We use `MolTrain` with regression task. The model automatically
downloads Uni-Mol pre-trained weights on first run.

In [ ]:
from unimol_tools import MolTrain

trainer = MolTrain(
    task='regression',
    data_type='molecule',
    epochs=50,
    batch_size=32,
    metrics='rmse',
    model_name='unimolv2',
    model_size='84m',
    save_path='./unimol_binding_model',
    remove_hs=False,
    target_cols='TARGET',
    learning_rate=1e-4,
)

trainer.fit(data='data/unimol_train.csv')

## 4. Evaluate on Test Set

In [ ]:
from unimol_tools import MolPredict
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr

predictor = MolPredict(load_model='./unimol_binding_model')
predictions = predictor.predict(data='data/unimol_test.csv')

y_true = test_df['TARGET'].values
y_pred = predictions.flatten()

rmse = mean_squared_error(y_true, y_pred, squared=False)
r, _ = pearsonr(y_true, y_pred)

print(f'Uni-Mol Test RMSE: {rmse:.3f}')
print(f'Uni-Mol Pearson r: {r:.3f}')

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_true, y_pred, alpha=0.5, s=20)
ax.plot([2, 12], [2, 12], 'r--', label='Ideal')
ax.set_xlabel('True pKa')
ax.set_ylabel('Predicted pKa')
ax.set_title(f'Uni-Mol v2 (84M) — RMSE={rmse:.3f}, r={r:.3f}')
ax.legend()
plt.tight_layout()
plt.savefig('unimol_scatter.png', dpi=150)
plt.show()

## 5. Export for Pipeline Integration

Download the `unimol_binding_model/` directory and place it in
your VLM project root. Then run the pipeline with:
```bash
python agent/pipeline.py --disease "Alzheimer's" --scoring unimol
```

In [ ]:
# Zip the model for download
!zip -r unimol_binding_model.zip unimol_binding_model/

# If running on Colab, this will show a download link
from google.colab import files
files.download('unimol_binding_model.zip')

## 6. Quick Inference Demo

Score new molecules using the fine-tuned model:

In [ ]:
# Score novel molecules
demo_smiles = [
    'CC(=O)OC1=CC=CC=C1C(=O)O',      # aspirin
    'CN1C=NC2=C1C(=O)N(C(=O)N2C)C',  # caffeine
    'c1ccc2c(c1)[nH]c1ccccc12',       # carbazole
]

demo_df = pd.DataFrame({'SMILES': demo_smiles})
demo_df.to_csv('data/demo_input.csv', index=False)

preds = predictor.predict(data='data/demo_input.csv')
for smi, pka in zip(demo_smiles, preds.flatten()):
    print(f'{smi:50s} → pKa = {pka:.2f}')